# MedASR Medical Speech Recognition with OpenVINO

[MedASR](https://huggingface.co/google/medasr) is a specialized Automatic Speech Recognition (ASR) model from Google, optimized for medical terminology. It is a CTC-based model built on top of the Conformer architecture.

In this notebook, we demonstrate how to:
- Load the MedASR model from HuggingFace
- Convert it to OpenVINO IR format for efficient inference
- Apply INT8 quantization using [NNCF](https://github.com/openvinotoolkit/nncf/) for model compression
- Compare accuracy and latency across PyTorch, FP16 OpenVINO, and INT8 OpenVINO versions



### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/medasr-medical-asr/medasr-medical-asr.ipynb" />
#### Table of contents:

- [1. Installation <a id="installation"></a>](#1.-Installation-<a-id="installation"></a>)
- [2. Login to HuggingFace <a id="login-huggingface"></a>](#2.-Login-to-HuggingFace-<a-id="login-huggingface"></a>)
- [3. Load Model <a id="load-model"></a>](#3.-Load-Model-<a-id="load-model"></a>)
- [4. Prepare Audio Data <a id="prepare-audio"></a>](#4.-Prepare-Audio-Data-<a-id="prepare-audio"></a>)
- [5. PyTorch Inference <a id="pytorch-inference"></a>](#5.-PyTorch-Inference-<a-id="pytorch-inference"></a>)
- [6. Convert to OpenVINO FP16 <a id="convert-fp16"></a>](#6.-Convert-to-OpenVINO-FP16-<a-id="convert-fp16"></a>)
- [7. INT8 Quantization <a id="int8-quantization"></a>](#7.-INT8-Quantization-<a-id="int8-quantization"></a>)
- [8. Select Inference Device <a id="select-device"></a>](#8.-Select-Inference-Device-<a-id="select-device"></a>)
- [9. Accuracy Comparison <a id="accuracy-comparison"></a>](#9.-Accuracy-Comparison-<a-id="accuracy-comparison"></a>)
- [10. Performance Benchmarking <a id="benchmarking"></a>](#10.-Performance-Benchmarking-<a-id="benchmarking"></a>)
- [Summary](#Summary)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## 1. Installation <a id="installation"></a>
[back to top ⬆️](#Table-of-contents:)

Install required packages for OpenVINO, NNCF, and model dependencies.

In [38]:
%pip install -q "openvino>=2026.1.0" "nncf>=3.0.0" "torch>=2.10" "transformers>=5.0.0" "librosa" "soundfile" "huggingface_hub" "matplotlib" "numpy" "ipywidgets"

import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("medasr-medical-asr.ipynb")

Note: you may need to restart the kernel to use updated packages.


## 2. Login to HuggingFace <a id="login-huggingface"></a>
[back to top ⬆️](#Table-of-contents:)

To run the model, you must be a registered user in 🤗 [Hugging Face Hub](https://huggingface.co/). 

The MedASR model is gated and requires you to:
1. Visit the [MedASR model card](https://huggingface.co/google/medasr)
2. Carefully read the terms of usage
3. Click the accept button to agree to the license

You will need to use an access token for the code below to run. For more information on access tokens, refer to [this section of the documentation](https://huggingface.co/docs/hub/security-tokens).

You can login to Hugging Face Hub using the following code:

In [39]:
# Login to HuggingFace Hub to get access to the pretrained model
# Uncomment the lines below if you need to authenticate

from huggingface_hub import whoami

try:
    whoami()
    print("Authorization token already provided")
except OSError:
    print("Please set HF_TOKEN environment variable or run: huggingface-cli login")
    # from huggingface_hub import notebook_login
    # notebook_login()

Authorization token already provided


## 3. Load Model <a id="load-model"></a>
[back to top ⬆️](#Table-of-contents:)

Load Google's MedASR model from HuggingFace. This is a CTC-based ASR model optimized for medical terminology.

In [40]:
from transformers import pipeline
import huggingface_hub
import librosa
import numpy as np
import torch
from pathlib import Path
import time

MODEL_ID = "google/medasr"
print(f"Loading model: {MODEL_ID}")

# Load model using pipeline
pipe = pipeline("automatic-speech-recognition", model=MODEL_ID, trust_remote_code=True)

# Extract model components
model = pipe.model
feature_extractor = pipe.feature_extractor
tokenizer = pipe.tokenizer

print(f"✓ Model loaded: {type(model).__name__}")
print(f"✓ Feature extractor: {type(feature_extractor).__name__}")
print(f"✓ Tokenizer vocab size: {tokenizer.vocab_size}")

Loading model: google/medasr


Loading weights:   0%|          | 0/368 [00:00<?, ?it/s]

✓ Model loaded: LasrForCTC
✓ Feature extractor: LasrFeatureExtractor
✓ Tokenizer vocab size: 512


## 4. Prepare Audio Data <a id="prepare-audio"></a>
[back to top ⬆️](#Table-of-contents:)

Download test audio and prepare it for model conversion. We use **10-second audio** for optimal GPU performance.

- Creates model with shape `[1, 998, 128]`
- Longer audio can be processed via chunking

In [41]:
# Download test audio from HuggingFace
audio_file = huggingface_hub.hf_hub_download("google/medasr", "test_audio.wav")
speech_full, sample_rate = librosa.load(audio_file, sr=16000)

print(f"Full audio duration: {len(speech_full) / sample_rate:.2f} seconds")

# Use 10s audio for optimal model shape
OPTIMAL_DURATION = 10.0
speech_10s = speech_full[: int(OPTIMAL_DURATION * sample_rate)]

print(f"Optimized audio duration: {len(speech_10s) / sample_rate:.2f} seconds")
print(f"Sample rate: {sample_rate} Hz")

# Extract features for model conversion
inputs = feature_extractor(speech_10s, sampling_rate=sample_rate, return_tensors="pt", padding=True, return_attention_mask=True)

input_features = inputs.input_features
attention_mask = inputs.attention_mask

SEQ_LEN = input_features.shape[1]
FEATURE_DIM = input_features.shape[2]

print(f"\n\u2713 Input features shape: {input_features.shape}")
print(f"\u2713 Attention mask shape: {attention_mask.shape}")
print(f"\u2713 Model will be created with static shape: [1, {SEQ_LEN}, {FEATURE_DIM}]")

Full audio duration: 43.80 seconds
Optimized audio duration: 10.00 seconds
Sample rate: 16000 Hz

✓ Input features shape: torch.Size([1, 998, 128])
✓ Attention mask shape: torch.Size([1, 998])
✓ Model will be created with static shape: [1, 998, 128]


## 5. PyTorch Inference <a id="pytorch-inference"></a>
[back to top ⬆️](#Table-of-contents:)

Run inference with PyTorch model to establish baseline accuracy.

In [42]:
# PyTorch inference
model.eval()
with torch.no_grad():
    pt_outputs = model(input_features, attention_mask=attention_mask)
    pt_logits = pt_outputs.logits.numpy()
    pt_ids = np.argmax(pt_logits, axis=-1)
    pt_transcription = tokenizer.batch_decode(pt_ids)[0]

print("PyTorch Inference Results:")
print(f"Transcription: {pt_transcription}")
print(f"Logits shape: {pt_logits.shape}")
print(f"Logits range: [{pt_logits.min():.2f}, {pt_logits.max():.2f}]")

PyTorch Inference Results:
Transcription: [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe</s>
Logits shape: (1, 247, 512)
Logits range: [-26.49, 24.95]


## 6. Convert to OpenVINO FP16 <a id="convert-fp16"></a>
[back to top ⬆️](#Table-of-contents:)

Convert the PyTorch model to OpenVINO IR format using `torch.export` and `ov.convert_model`.
The model is saved with FP16 weight compression for reduced size.

In [43]:
import openvino as ov
import os

MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

FP16_MODEL_PATH = MODEL_DIR / "medasr_fp16.xml"

if not FP16_MODEL_PATH.exists():
    # Create model wrapper for clean export
    class MedASRWrapper(torch.nn.Module):
        def __init__(self, model):
            super().__init__()
            self.model = model

        def forward(self, input_features, attention_mask):
            outputs = self.model(input_features=input_features, attention_mask=attention_mask)
            return outputs.logits

    wrapped_model = MedASRWrapper(model)
    wrapped_model.eval()

    print("Converting PyTorch model to OpenVINO IR...")
    print(f"Input shape: {input_features.shape}")

    with torch.no_grad():
        # Export using torch.export
        exported = torch.export.export(wrapped_model, (input_features, attention_mask))
        print("\u2713 Model exported with torch.export")

        # Convert to OpenVINO
        ov_model = ov.convert_model(exported)

        # Reshape to static shape for optimal performance
        ov_model.reshape({"input_features": [1, SEQ_LEN, FEATURE_DIM], "attention_mask": [1, SEQ_LEN]})
        print(f"\u2713 Model reshaped to static: [1, {SEQ_LEN}, {FEATURE_DIM}]")

    # Save FP16 model
    ov.save_model(ov_model, FP16_MODEL_PATH, compress_to_fp16=True)
    print(f"\n\u2713 FP16 model saved: {FP16_MODEL_PATH}")
else:
    print(f"FP16 model already exists: {FP16_MODEL_PATH}")
    ov_model = ov.Core().read_model(FP16_MODEL_PATH)

fp16_size = (os.path.getsize(FP16_MODEL_PATH) + os.path.getsize(FP16_MODEL_PATH.with_suffix(".bin"))) / 1024 / 1024
print(f"\u2713 Model size: {fp16_size:.2f} MB")

# Verify model inputs
print("\nModel inputs:")
for inp in ov_model.inputs:
    print(f"  {inp.get_any_name()}: {inp.partial_shape}")

FP16 model already exists: model/medasr_fp16.xml
✓ Model size: 202.10 MB

Model inputs:
  input_features: [1,998,128]
  attention_mask: [1,998]


## 7. INT8 Quantization <a id="int8-quantization"></a>
[back to top ⬆️](#Table-of-contents:)

Quantize the model to INT8 using NNCF with **real audio data** for calibration.

**Key Settings:**
- `ModelType.TRANSFORMER` - Optimized for transformer models
- Real audio calibration data - Better accuracy than random data
- `fast_bias_correction` - Faster quantization with good results

In [44]:
import nncf
from nncf import Dataset

INT8_MODEL_PATH = MODEL_DIR / "medasr_int8.xml"

if not INT8_MODEL_PATH.exists():
    print("Preparing calibration data from real audio...")

    # Create calibration data from the test audio with variations
    calibration_data = []

    # Use the real audio features as base
    base_features = input_features.numpy().astype(np.float32)
    base_mask = attention_mask.numpy().astype(np.float32)

    # Add the original sample
    calibration_data.append({"input_features": base_features, "attention_mask": base_mask})

    # Create variations with realistic audio augmentations
    np.random.seed(42)
    for i in range(99):  # Total 100 calibration samples
        # Add small realistic noise (simulates different recording conditions)
        noise_level = np.random.uniform(0.001, 0.02)
        noisy_features = base_features + np.random.randn(*base_features.shape).astype(np.float32) * noise_level

        # Slight volume variation
        volume_scale = np.random.uniform(0.8, 1.2)
        noisy_features = noisy_features * volume_scale

        calibration_data.append({"input_features": noisy_features, "attention_mask": base_mask.copy()})

    print(f"\u2713 Created {len(calibration_data)} calibration samples")

    # Create NNCF dataset
    def transform_fn(data_item):
        return {"input_features": data_item["input_features"], "attention_mask": data_item["attention_mask"]}

    calibration_dataset = Dataset(calibration_data, transform_fn)

    print("\nQuantizing to INT8 with TRANSFORMER preset...")
    print("This may take a few minutes...")

    quantized_model = nncf.quantize(
        model=ov_model,
        calibration_dataset=calibration_dataset,
        subset_size=min(100, len(calibration_data)),
        model_type=nncf.ModelType.TRANSFORMER,
        fast_bias_correction=True,
    )

    print("\u2713 Quantization complete!")

    # Verify INT8 model inputs
    print("\nQuantized model inputs:")
    for inp in quantized_model.inputs:
        print(f"  {inp.get_any_name()}: {inp.partial_shape}")

    # Save INT8 model
    ov.save_model(quantized_model, INT8_MODEL_PATH)
    print(f"\n\u2713 INT8 model saved: {INT8_MODEL_PATH}")
else:
    print(f"INT8 model already exists: {INT8_MODEL_PATH}")
    quantized_model = ov.Core().read_model(INT8_MODEL_PATH)

int8_size = (os.path.getsize(INT8_MODEL_PATH) + os.path.getsize(INT8_MODEL_PATH.with_suffix(".bin"))) / 1024 / 1024
print(f"\u2713 Model size: {int8_size:.2f} MB")
print(f"\u2713 Compression ratio: {fp16_size / int8_size:.2f}x")

INT8 model already exists: model/medasr_int8.xml
✓ Model size: 103.51 MB
✓ Compression ratio: 1.95x


In [45]:
# Display quantization statistics
op_types = {}
for op in quantized_model.get_ops():
    op_type = op.get_type_name()
    op_types[op_type] = op_types.get(op_type, 0) + 1

print("Quantized model statistics:")
print(f"  FakeQuantize ops: {op_types.get('FakeQuantize', 0)}")
print(f"  Convolution ops: {op_types.get('Convolution', 0)}")
print(f"  MatMul ops: {op_types.get('MatMul', 0)}")
print(f"  Total ops: {sum(op_types.values())}")

Quantized model statistics:
  FakeQuantize ops: 192
  Convolution ops: 37
  MatMul ops: 138
  Total ops: 4054


## 8. Select Inference Device <a id="select-device"></a>
[back to top ⬆️](#Table-of-contents:)

Select the device for running inference. The model can run on CPU, GPU, or other available OpenVINO devices.

In [46]:
from notebook_utils import device_widget

device = device_widget()

device

Dropdown(description='Device:', index=3, options=('CPU', 'GPU', 'NPU', 'AUTO'), value='AUTO')

## 9. Accuracy Comparison <a id="accuracy-comparison"></a>
[back to top ⬆️](#Table-of-contents:)

Compare accuracy of PyTorch, FP16, and INT8 models to ensure quantization quality.

In [47]:
import openvino as ov

print("=" * 70)
print("ACCURACY COMPARISON: PyTorch vs FP16 vs INT8")
print("=" * 70)

core = ov.Core()

# Prepare input data
np_features = input_features.numpy().astype(np.float32)
np_mask = attention_mask.numpy().astype(np.float32)

# Compile models on the selected device
print(f"\nCompiling models for {device.value}...")
config = {"PERFORMANCE_HINT": "LATENCY"}
if device.value != "CPU":
    config["INFERENCE_PRECISION_HINT"] = "f32"

fp16_compiled = core.compile_model(FP16_MODEL_PATH, device.value, config)
int8_compiled = core.compile_model(INT8_MODEL_PATH, device.value, config)

# FP16 inference
fp16_out = fp16_compiled({"input_features": np_features, "attention_mask": np_mask})
fp16_logits = fp16_out[0]
fp16_ids = np.argmax(fp16_logits, axis=-1)
fp16_text = tokenizer.batch_decode(fp16_ids)[0]

# INT8 inference
int8_out = int8_compiled({"input_features": np_features, "attention_mask": np_mask})
int8_logits = int8_out[0]
int8_ids = np.argmax(int8_logits, axis=-1)
int8_text = tokenizer.batch_decode(int8_ids)[0]

print("\n--- Transcriptions ---")
print(f"PyTorch: {pt_transcription}")
print(f"FP16:    {fp16_text}")
print(f"INT8:    {int8_text}")


# Calculate accuracy metrics
def calculate_accuracy(ref_ids, hyp_ids):
    return np.mean(ref_ids == hyp_ids) * 100


fp16_vs_pytorch = calculate_accuracy(pt_ids, fp16_ids)
int8_vs_pytorch = calculate_accuracy(pt_ids, int8_ids)
int8_vs_fp16 = calculate_accuracy(fp16_ids, int8_ids)

print("\n--- Token Match Accuracy ---")
print(f"FP16 vs PyTorch: {fp16_vs_pytorch:.2f}%")
print(f"INT8 vs PyTorch: {int8_vs_pytorch:.2f}%")
print(f"INT8 vs FP16:    {int8_vs_fp16:.2f}%")

# Logit correlation
fp16_corr = np.corrcoef(pt_logits.flatten(), fp16_logits.flatten())[0, 1]
int8_corr = np.corrcoef(pt_logits.flatten(), int8_logits.flatten())[0, 1]

print("\n--- Logit Correlation ---")
print(f"FP16 vs PyTorch: {fp16_corr:.6f}")
print(f"INT8 vs PyTorch: {int8_corr:.6f}")

print("\n" + "=" * 70)
if fp16_vs_pytorch >= 99.0 and int8_vs_pytorch >= 95.0:
    print("\u2713 ACCURACY CHECK PASSED")
else:
    print("\u26a0 ACCURACY CHECK: Review results above")
print("=" * 70)

ACCURACY COMPARISON: PyTorch vs FP16 vs INT8

Compiling models for AUTO...

--- Transcriptions ---
PyTorch: [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe</s>
FP16:    [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe</s>
INT8:    [EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNiQe</s>

--- Token Match Accuracy ---
FP16 vs PyTorch: 99.60%
INT8 vs PyTorch: 99.19%
INT8 vs FP16:    98.79%

--- Logit Correlation ---
FP16 vs PyTorch: 1.000000
INT8 vs PyTorch: 0.996420

✓ ACCURACY CHECK PASSED


## 10. Performance Benchmarking <a id="benchmarking"></a>
[back to top ⬆️](#Table-of-contents:)

Benchmark FP16 and INT8 models on the selected device.

In [48]:
print("=" * 70)
print("PERFORMANCE BENCHMARKING")
print("=" * 70)

core = ov.Core()
print(f"Benchmarking on device: {device.value}")

# Device-specific config
config = {"PERFORMANCE_HINT": "LATENCY"}
if device.value != "CPU":
    config["INFERENCE_PRECISION_HINT"] = "f32"

# FP16 benchmark
fp16_model = core.compile_model(FP16_MODEL_PATH, device.value, config)

# Warmup
for _ in range(10):
    fp16_model({"input_features": np_features, "attention_mask": np_mask})

# Benchmark
fp16_latencies = []
for _ in range(100):
    start = time.time()
    fp16_model({"input_features": np_features, "attention_mask": np_mask})
    fp16_latencies.append((time.time() - start) * 1000)

fp16_median = np.median(fp16_latencies)
fp16_min = np.min(fp16_latencies)

# INT8 benchmark
int8_model = core.compile_model(INT8_MODEL_PATH, device.value, config)

# Warmup
for _ in range(10):
    int8_model({"input_features": np_features, "attention_mask": np_mask})

# Benchmark
int8_latencies = []
for _ in range(100):
    start = time.time()
    int8_model({"input_features": np_features, "attention_mask": np_mask})
    int8_latencies.append((time.time() - start) * 1000)

int8_median = np.median(int8_latencies)
int8_min = np.min(int8_latencies)

speedup = fp16_median / int8_median

print(f"\nFP16: {fp16_median:.2f}ms (min: {fp16_min:.2f}ms)")
print(f"INT8: {int8_median:.2f}ms (min: {int8_min:.2f}ms)")
print(f"Speedup: {speedup:.2f}x")

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print("\nModel sizes:")
print(f"  FP16: {fp16_size:.2f} MB")
print(f"  INT8: {int8_size:.2f} MB")
print(f"  Compression: {fp16_size / int8_size:.2f}x")

print("\nAccuracy (vs PyTorch):")
print(f"  FP16: {fp16_vs_pytorch:.2f}%")
print(f"  INT8: {int8_vs_pytorch:.2f}%")

print(f"\nLatency ({device.value}):")
print(f"  FP16: {fp16_median:.2f}ms")
print(f"  INT8: {int8_median:.2f}ms")
print(f"  Speedup: {speedup:.2f}x")
print("=" * 70)

PERFORMANCE BENCHMARKING
Benchmarking on device: AUTO

FP16: 47.35ms (min: 44.10ms)
INT8: 6.66ms (min: 6.30ms)
Speedup: 7.11x

SUMMARY

Model sizes:
  FP16: 202.10 MB
  INT8: 103.51 MB
  Compression: 1.95x

Accuracy (vs PyTorch):
  FP16: 99.60%
  INT8: 99.19%

Latency (AUTO):
  FP16: 47.35ms
  INT8: 6.66ms
  Speedup: 7.11x


## Summary
[back to top ⬆️](#Table-of-contents:)

This notebook demonstrated the full workflow for optimizing Google's MedASR model with OpenVINO:

- Loaded the MedASR model from HuggingFace
- Converted to OpenVINO IR format with FP16 weight compression
- Applied INT8 quantization using NNCF with real audio calibration data
- Compared accuracy and latency across PyTorch, FP16, and INT8

**Generated Models:**
- `model/medasr_fp16.xml` - FP16 model for inference
- `model/medasr_int8.xml` - INT8 quantized model with compression

**Links:**
- [MedASR Model Card on HuggingFace](https://huggingface.co/google/medasr)
- [OpenVINO Documentation](https://docs.openvino.ai/)
- [NNCF Documentation](https://github.com/openvinotoolkit/nncf/)